# Word2Vec: From One-Hot Encoding to Embeddings
## Understanding the Complete Pipeline

This notebook demonstrates:
1. **One-Hot Encoding** - Traditional sparse representation
2. **Word2Vec Training** - Learning dense embeddings from scratch
3. **Comparison** - One-hot vs. learned embeddings
4. **Visualization** - Understanding the learned representations

We'll use the classic sentence: **"The quick brown fox jumping over the lazy dog"**

In [ ]:
# Install required packages
!pip install -q gensim matplotlib numpy pandas seaborn scikit-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## Step 1: Prepare the Data

We'll start with our sample sentence and create a small corpus for training.

In [ ]:
# Sample sentence
sample_sentence = "The quick brown fox jumping over the lazy dog"

print("Original Sentence:")
print(f"'{sample_sentence}'")
print(f"\nLength: {len(sample_sentence)} characters")

In [ ]:
# Tokenize the sentence
tokens = sample_sentence.lower().split()

print("Tokenized:")
print(tokens)
print(f"\nNumber of tokens: {len(tokens)}")
print(f"Unique words: {len(set(tokens))}")

In [ ]:
# Create vocabulary
vocabulary = sorted(set(tokens))
vocab_size = len(vocabulary)

# Create word to index and index to word mappings
word_to_idx = {word: idx for idx, word in enumerate(vocabulary)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

print("Vocabulary (alphabetically sorted):")
for idx, word in enumerate(vocabulary):
    print(f"{idx}: {word}")

print(f"\nVocabulary size: {vocab_size}")

## Step 2: One-Hot Encoding

One-hot encoding represents each word as a vector of length V (vocabulary size), where:
- All elements are 0, except
- One element is 1 at the position corresponding to the word

**Example:** If vocabulary = [brown, dog, fox, ...], then:
- 'brown' = [1, 0, 0, 0, 0, 0, 0, 0]
- 'dog' = [0, 1, 0, 0, 0, 0, 0, 0]
- 'fox' = [0, 0, 1, 0, 0, 0, 0, 0]

In [ ]:
def create_one_hot(word, word_to_idx, vocab_size):
    """
    Create one-hot encoding for a word
    
    Args:
        word: string, the word to encode
        word_to_idx: dict, mapping from word to index
        vocab_size: int, size of vocabulary
    
    Returns:
        numpy array of shape (vocab_size,) with one-hot encoding
    """
    one_hot = np.zeros(vocab_size)
    idx = word_to_idx[word]
    one_hot[idx] = 1
    return one_hot

In [ ]:
# Create one-hot encodings for all words in our sentence
one_hot_encodings = {}

for word in vocabulary:
    one_hot_encodings[word] = create_one_hot(word, word_to_idx, vocab_size)

print("One-Hot Encodings:")
print("="*50)
for word in vocabulary[:3]:  # Show first 3 words
    encoding = one_hot_encodings[word]
    print(f"\n'{word}':")
    print(f"Vector: {encoding}")
    print(f"Shape: {encoding.shape}")
    print(f"Index of '1': {np.argmax(encoding)}")

In [ ]:
# Visualize one-hot encodings as a matrix
one_hot_matrix = np.array([one_hot_encodings[word] for word in vocabulary])

plt.figure(figsize=(10, 8))
plt.imshow(one_hot_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Value')
plt.xlabel('Dimension', fontsize=12)
plt.ylabel('Word', fontsize=12)
plt.title('One-Hot Encoding Matrix\n(Each row is a word, each column is a dimension)', 
          fontsize=14, fontweight='bold')
plt.yticks(range(vocab_size), vocabulary)
plt.xticks(range(vocab_size), range(vocab_size))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"One-hot matrix shape: {one_hot_matrix.shape}")
print(f"Total elements: {one_hot_matrix.size}")
print(f"Non-zero elements: {np.count_nonzero(one_hot_matrix)}")
print(f"Sparsity: {(1 - np.count_nonzero(one_hot_matrix) / one_hot_matrix.size) * 100:.1f}%")

### Problems with One-Hot Encoding

Let's demonstrate the key limitations:

In [ ]:
# Problem 1: No semantic similarity
def cosine_similarity(v1, v2):
    """Calculate cosine similarity between two vectors"""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

# Compare semantic pairs
word_pairs = [
    ('fox', 'dog'),      # Both animals
    ('quick', 'lazy'),   # Both adjectives (opposites)
    ('brown', 'fox'),    # Unrelated
    ('the', 'over')      # Both function words
]

print("Cosine Similarity using One-Hot Encoding:")
print("="*50)
for word1, word2 in word_pairs:
    vec1 = one_hot_encodings[word1]
    vec2 = one_hot_encodings[word2]
    similarity = cosine_similarity(vec1, vec2)
    print(f"similarity('{word1}', '{word2}') = {similarity:.4f}")

print("\n💡 Problem: All word pairs have similarity = 0.0")
print("   One-hot encoding treats all words as equally different!")

In [ ]:
# Problem 2: High dimensionality
print("Dimensionality Analysis:")
print("="*50)
print(f"Vocabulary size: {vocab_size} words")
print(f"One-hot vector dimension: {vocab_size}")
print(f"\nFor real applications:")
print(f"  - 10,000 word vocabulary → 10,000 dimensions")
print(f"  - 100,000 word vocabulary → 100,000 dimensions")
print(f"\n💡 Problem: Dimension grows with vocabulary size!")
print(f"   This makes models computationally expensive.")

In [ ]:
# Problem 3: Extreme sparsity
print("Sparsity Analysis:")
print("="*50)

# For our example
total_elements = one_hot_matrix.size
non_zero = np.count_nonzero(one_hot_matrix)
sparsity = (1 - non_zero / total_elements) * 100

print(f"Matrix size: {one_hot_matrix.shape}")
print(f"Total elements: {total_elements}")
print(f"Non-zero elements: {non_zero}")
print(f"Sparsity: {sparsity:.1f}%")
print(f"\n💡 Problem: {sparsity:.1f}% of values are zero!")
print(f"   Wastes memory and computation.")

## Step 3: Creating Training Data for Word2Vec

To train Word2Vec properly, we need more data. We'll create additional sentences using our vocabulary.

In [ ]:
# Create a corpus with multiple sentences
# This helps Word2Vec learn better word relationships
corpus = [
    "the quick brown fox jumping over the lazy dog",
    "the lazy dog sleeping over there",
    "the quick fox running fast",
    "the brown dog jumping high",
    "quick brown fox and lazy brown dog",
    "the fox jumping over the dog",
    "lazy dog and quick fox playing",
    "brown fox running over the grass",
    "the dog and the fox are friends",
    "quick animals like fox and dog",
    "the brown lazy cat sleeping",
    "quick cat jumping over the fence",
    "the cat and dog playing together",
    "lazy cat and lazy dog resting",
    "quick movements of the fox",
]

print("Training Corpus:")
print("="*50)
for i, sentence in enumerate(corpus, 1):
    print(f"{i:2d}. {sentence}")

print(f"\nTotal sentences: {len(corpus)}")

In [ ]:
# Tokenize all sentences
tokenized_corpus = [sentence.lower().split() for sentence in corpus]

print("Tokenized Corpus (first 3 sentences):")
for i, tokens in enumerate(tokenized_corpus[:3], 1):
    print(f"{i}. {tokens}")

# Calculate corpus statistics
all_words = [word for sentence in tokenized_corpus for word in sentence]
word_freq = Counter(all_words)

print(f"\nCorpus Statistics:")
print(f"Total words: {len(all_words)}")
print(f"Unique words: {len(word_freq)}")
print(f"\nTop 10 most frequent words:")
for word, count in word_freq.most_common(10):
    print(f"  '{word}': {count} times")

## Step 4: Training Word2Vec Model

Now we'll train a Word2Vec model from scratch on our corpus.

### Word2Vec Parameters:
- **vector_size**: Dimension of word embeddings (we'll use 50, much smaller than one-hot!)
- **window**: Context window size (how many words around target word)
- **min_count**: Ignore words with frequency less than this
- **sg**: Training algorithm (0=CBOW, 1=Skip-gram)
- **epochs**: Number of training iterations

In [ ]:
# Train Word2Vec model
print("Training Word2Vec model...\n")

model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,      # Embedding dimension (50 vs 8+ for one-hot!)
    window=3,            # Context window: 3 words before and after
    min_count=1,         # Include all words (even if appear once)
    sg=1,                # Use Skip-gram (1) instead of CBOW (0)
    epochs=100,          # Train for 100 iterations
    seed=42              # For reproducibility
)

print("✓ Training complete!")
print(f"\nModel Information:")
print(f"Vocabulary size: {len(model.wv)}")
print(f"Vector dimension: {model.wv.vector_size}")
print(f"Training algorithm: {'Skip-gram' if model.sg else 'CBOW'}")

## Step 5: Exploring Word2Vec Embeddings

Let's examine the learned embeddings and compare them with one-hot encoding.

In [ ]:
# Get word vector
word = 'fox'
word2vec_vector = model.wv[word]
onehot_vector = one_hot_encodings[word]

print(f"Comparison for word: '{word}'")
print("="*70)

print(f"\nOne-Hot Encoding:")
print(f"  Shape: {onehot_vector.shape}")
print(f"  Vector: {onehot_vector}")
print(f"  Non-zero elements: {np.count_nonzero(onehot_vector)}")

print(f"\nWord2Vec Embedding:")
print(f"  Shape: {word2vec_vector.shape}")
print(f"  First 10 dimensions: {word2vec_vector[:10]}")
print(f"  Non-zero elements: {np.count_nonzero(word2vec_vector)}")
print(f"  Min value: {word2vec_vector.min():.4f}")
print(f"  Max value: {word2vec_vector.max():.4f}")
print(f"  Mean value: {word2vec_vector.mean():.4f}")

In [ ]:
# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# One-hot encoding
axes[0].bar(range(len(onehot_vector)), onehot_vector, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Value')
axes[0].set_title(f'One-Hot Encoding for "{word}"\n({len(onehot_vector)} dimensions, sparse)', 
                  fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Word2Vec embedding
axes[1].bar(range(len(word2vec_vector)), word2vec_vector, color='coral', alpha=0.7)
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Value')
axes[1].set_title(f'Word2Vec Embedding for "{word}"\n({len(word2vec_vector)} dimensions, dense)', 
                  fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Semantic Similarity with Word2Vec

The key advantage: Word2Vec captures semantic relationships!

In [ ]:
# Compare semantic similarities
word_pairs = [
    ('fox', 'dog'),
    ('quick', 'lazy'),
    ('brown', 'fox'),
    ('jumping', 'running'),
    ('cat', 'dog'),
    ('the', 'over')
]

print("Semantic Similarity Comparison:")
print("="*70)
print(f"{'Word Pair':<20} {'One-Hot':<15} {'Word2Vec':<15} {'Difference'}")
print("-"*70)

for word1, word2 in word_pairs:
    # One-hot similarity
    if word1 in one_hot_encodings and word2 in one_hot_encodings:
        onehot_sim = cosine_similarity(
            one_hot_encodings[word1], 
            one_hot_encodings[word2]
        )
    else:
        onehot_sim = 0.0
    
    # Word2Vec similarity
    try:
        w2v_sim = model.wv.similarity(word1, word2)
    except:
        w2v_sim = 0.0
    
    diff = w2v_sim - onehot_sim
    
    print(f"{word1+'-'+word2:<20} {onehot_sim:>6.4f}        {w2v_sim:>6.4f}        {diff:>+6.4f}")

print("\n💡 Word2Vec captures meaningful similarities!")
print("   Similar words (fox-dog, cat-dog) have higher similarity scores.")

In [ ]:
# Find most similar words
test_words = ['fox', 'dog', 'quick', 'lazy', 'jumping']

print("Most Similar Words (Word2Vec):")
print("="*50)

for word in test_words:
    if word in model.wv:
        print(f"\nWords similar to '{word}':")
        similar_words = model.wv.most_similar(word, topn=5)
        for similar_word, score in similar_words:
            print(f"  {similar_word:15s}: {score:.4f}")

## Step 7: Visualization of Embeddings

Let's visualize the 50-dimensional Word2Vec embeddings in 2D space.

In [ ]:
# Get all word vectors
words = list(model.wv.index_to_key)
word_vectors = np.array([model.wv[word] for word in words])

print(f"Word vectors shape: {word_vectors.shape}")
print(f"Number of words: {len(words)}")
print(f"Embedding dimension: {word_vectors.shape[1]}")

In [ ]:
# Reduce dimensions using PCA
pca = PCA(n_components=2, random_state=42)
word_vectors_2d_pca = pca.fit_transform(word_vectors)

print("PCA Information:")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Visualize with PCA
plt.figure(figsize=(14, 10))

# Define categories for coloring
animals = ['fox', 'dog', 'cat']
adjectives = ['quick', 'lazy', 'brown']
actions = ['jumping', 'running', 'sleeping', 'playing', 'resting']
function_words = ['the', 'over', 'and', 'are', 'of']

# Plot with different colors for different categories
for i, word in enumerate(words):
    x, y = word_vectors_2d_pca[i]
    
    if word in animals:
        color = 'red'
        marker = 'o'
        size = 200
    elif word in adjectives:
        color = 'blue'
        marker = 's'
        size = 200
    elif word in actions:
        color = 'green'
        marker = '^'
        size = 200
    elif word in function_words:
        color = 'gray'
        marker = 'D'
        size = 100
    else:
        color = 'orange'
        marker = 'p'
        size = 150
    
    plt.scatter(x, y, c=color, marker=marker, s=size, alpha=0.6, edgecolors='black', linewidth=1)
    plt.annotate(word, (x, y), fontsize=11, fontweight='bold', 
                ha='center', va='bottom', alpha=0.9)

# Create legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='Animals'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='blue', markersize=10, label='Adjectives'),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='green', markersize=10, label='Actions'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='gray', markersize=8, label='Function words'),
    Line2D([0], [0], marker='p', color='w', markerfacecolor='orange', markersize=10, label='Other')
]
plt.legend(handles=legend_elements, loc='best', fontsize=10)

plt.title('Word2Vec Embeddings Visualization (PCA)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Notice how similar words cluster together!")
print("   - Animals (fox, dog, cat) are near each other")
print("   - Adjectives (quick, lazy, brown) form another cluster")
print("   - Action words (jumping, running) are grouped")

In [ ]:
# Alternative visualization with t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(words)-1))
word_vectors_2d_tsne = tsne.fit_transform(word_vectors)

plt.figure(figsize=(14, 10))

for i, word in enumerate(words):
    x, y = word_vectors_2d_tsne[i]
    
    if word in animals:
        color = 'red'
        marker = 'o'
        size = 200
    elif word in adjectives:
        color = 'blue'
        marker = 's'
        size = 200
    elif word in actions:
        color = 'green'
        marker = '^'
        size = 200
    elif word in function_words:
        color = 'gray'
        marker = 'D'
        size = 100
    else:
        color = 'orange'
        marker = 'p'
        size = 150
    
    plt.scatter(x, y, c=color, marker=marker, s=size, alpha=0.6, edgecolors='black', linewidth=1)
    plt.annotate(word, (x, y), fontsize=11, fontweight='bold', 
                ha='center', va='bottom', alpha=0.9)

plt.legend(handles=legend_elements, loc='best', fontsize=10)
plt.title('Word2Vec Embeddings Visualization (t-SNE)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("t-SNE often reveals different clustering patterns than PCA!")

## Step 8: Sentence Encoding Comparison

Let's encode our original sentence using both methods.

In [ ]:
# Original sentence
sentence = "the quick brown fox jumping over the lazy dog"
sentence_tokens = sentence.lower().split()

print(f"Original sentence: '{sentence}'")
print(f"Tokens: {sentence_tokens}")
print(f"Number of tokens: {len(sentence_tokens)}")

In [ ]:
# Method 1: One-hot encoding (concatenate all word vectors)
sentence_onehot = np.concatenate([one_hot_encodings.get(word, np.zeros(vocab_size)) 
                                   for word in sentence_tokens])

print("Sentence Encoding with One-Hot:")
print(f"  Dimension: {sentence_onehot.shape[0]}")
print(f"  Non-zero elements: {np.count_nonzero(sentence_onehot)}")
print(f"  Sparsity: {(1 - np.count_nonzero(sentence_onehot) / sentence_onehot.size) * 100:.1f}%")
print(f"  Memory (approx): {sentence_onehot.nbytes} bytes")

In [ ]:
# Method 2: Word2Vec (average all word vectors)
sentence_word2vec = np.mean([model.wv[word] for word in sentence_tokens 
                             if word in model.wv], axis=0)

print("Sentence Encoding with Word2Vec (averaged):")
print(f"  Dimension: {sentence_word2vec.shape[0]}")
print(f"  Non-zero elements: {np.count_nonzero(sentence_word2vec)}")
print(f"  Sparsity: {(1 - np.count_nonzero(sentence_word2vec) / sentence_word2vec.size) * 100:.1f}%")
print(f"  Memory (approx): {sentence_word2vec.nbytes} bytes")

In [ ]:
# Comparison table
comparison_data = {
    'Method': ['One-Hot', 'Word2Vec'],
    'Dimensions': [sentence_onehot.shape[0], sentence_word2vec.shape[0]],
    'Sparsity': [
        f"{(1 - np.count_nonzero(sentence_onehot) / sentence_onehot.size) * 100:.1f}%",
        f"{(1 - np.count_nonzero(sentence_word2vec) / sentence_word2vec.size) * 100:.1f}%"
    ],
    'Memory (bytes)': [sentence_onehot.nbytes, sentence_word2vec.nbytes],
    'Captures Semantics': ['No', 'Yes']
}

df_comparison = pd.DataFrame(comparison_data)
print("\nSentence Encoding Comparison:")
print("="*70)
print(df_comparison.to_string(index=False))

print("\n💡 Key Takeaway:")
print(f"   Word2Vec uses {sentence_word2vec.shape[0]} dimensions vs {sentence_onehot.shape[0]} for one-hot")
print(f"   That's a {sentence_onehot.shape[0] / sentence_word2vec.shape[0]:.1f}x reduction!")
print(f"   Plus, Word2Vec captures semantic meaning!")

## Step 9: Understanding the Training Process

Let's visualize how Word2Vec learns from context.

In [ ]:
# Show context windows for training
example_sentence = "the quick brown fox jumping"
example_tokens = example_sentence.split()
window_size = 2

print("Word2Vec Training: Context Windows (Skip-gram)")
print("="*70)
print(f"Sentence: '{example_sentence}'")
print(f"Window size: {window_size}\n")

for i, target_word in enumerate(example_tokens):
    # Get context words within window
    start = max(0, i - window_size)
    end = min(len(example_tokens), i + window_size + 1)
    context = example_tokens[start:i] + example_tokens[i+1:end]
    
    print(f"Target: '{target_word}'")
    print(f"  Context: {context}")
    print(f"  Skip-gram trains: '{target_word}' → '{ctx}' for each ctx in context\n")

In [ ]:
# Demonstrate the learning objective
print("Skip-gram Learning Objective:")
print("="*70)
print("For target word 'fox' with context ['quick', 'brown', 'jumping']:\n")
print("The model learns to:")
print("  1. Given 'fox', predict 'quick'")
print("  2. Given 'fox', predict 'brown'")
print("  3. Given 'fox', predict 'jumping'")
print("\nThis forces the model to encode semantic information:")
print("  - 'fox' learns to be similar to 'dog' (similar contexts)")
print("  - 'quick' learns to be similar to 'fast' (similar contexts)")
print("  - Words that appear together get similar embeddings")

## Step 10: Summary and Key Insights

In [ ]:
# Create comprehensive summary
print("\n" + "="*70)
print("SUMMARY: One-Hot Encoding vs Word2Vec")
print("="*70)

summary_data = {
    'Aspect': [
        'Dimensionality',
        'Density',
        'Semantic Meaning',
        'Memory Efficiency',
        'Training Required',
        'Similarity Measure',
        'Best Use Case'
    ],
    'One-Hot Encoding': [
        f'{vocab_size} (= vocab size)',
        f'{(1 - np.count_nonzero(one_hot_matrix) / one_hot_matrix.size) * 100:.0f}% sparse',
        'No - all words equally different',
        'Poor - mostly zeros',
        'No',
        'Always 0 (orthogonal)',
        'Simple baselines, interpretability'
    ],
    'Word2Vec': [
        '50 (user-defined, typically 50-300)',
        '~0% sparse (dense)',
        'Yes - captures context',
        'Excellent - compact',
        'Yes - from corpus',
        'Meaningful (0 to 1)',
        'Most NLP tasks, transfer learning'
    ]
}

df_summary = pd.DataFrame(summary_data)
print("\n" + df_summary.to_string(index=False))

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)
print("\n1. DIMENSIONALITY REDUCTION:")
print(f"   One-hot: {vocab_size} dimensions")
print(f"   Word2Vec: 50 dimensions")
print(f"   Reduction: {vocab_size/50:.1f}x smaller!")

print("\n2. SEMANTIC UNDERSTANDING:")
print("   One-hot treats 'fox' and 'dog' as completely different")
print("   Word2Vec learns they are similar (both animals)")

print("\n3. SCALABILITY:")
print("   One-hot grows with vocabulary (10,000 words = 10,000 dims)")
print("   Word2Vec stays fixed (can be 50-300 dims for any vocab size)")

print("\n4. TRAINING:")
print("   One-hot: No training needed (simple lookup)")
print("   Word2Vec: Requires corpus and training time")
print("   Trade-off: Simplicity vs Expressiveness")

print("\n" + "="*70)

## Exercises for Students

Try the following to deepen your understanding:

1. **Modify the corpus**: Add 5-10 more sentences and retrain Word2Vec. How do the similarities change?

2. **Experiment with parameters**:
   - Try different `vector_size` (10, 50, 100, 200)
   - Try different `window` sizes (1, 3, 5, 10)
   - Compare CBOW (`sg=0`) vs Skip-gram (`sg=1`)

3. **Similarity exploration**:
   - Find words similar to 'cat'
   - Try word arithmetic: 'quick' - 'fast' + 'slow'
   - Which word pairs have highest similarity?

4. **Visualization**:
   - Try 3D visualization using PCA with 3 components
   - Color words by part of speech (nouns, verbs, adjectives)

5. **Real-world application**:
   - Load a larger corpus (news articles, books)
   - Train Word2Vec on it
   - Compare with pre-trained embeddings (Google News)

## Additional Resources

- **Original Word2Vec Paper**: "Efficient Estimation of Word Representations in Vector Space" (Mikolov et al., 2013)
- **Pre-trained Models**: Google News Word2Vec, GloVe, fastText
- **Advanced Topics**: 
  - Contextual embeddings (ELMo, BERT)
  - Subword embeddings (fastText)
  - Multilingual embeddings